# 01-CROHME Dataset Exploration

This notebook performs the **initial exploration and structural analysis of the CROHME handwritten mathematical expression dataset** used in this project.

The goal is to understand the raw **InkML files** and their corresponding **LaTeX annotations** before image generation, preprocessing, model development, and training.

### Working Objectives

The notebook will:

1. Identify and inspect the available CROHME dataset splits.
2. Discover and parse InkML expression files and their LaTeX annotations.
3. Analyze the number of strokes and points in each expression.
4. Measure geometric properties such as width, height, and aspect ratio.
5. Analyze LaTeX sequence length and basic formatting characteristics.
6. Visualize distributions of aspect ratio, stroke count, and LaTeX length.
7. Identify unusually large or complex expressions relevant to later preprocessing.
8. Validate basic properties of the LaTeX annotations, including dollar-sign delimiters.

### Dataset Representation

Each CROHME expression is stored in **InkML** format as a collection of handwritten strokes, with each stroke represented by sampled trajectory points.

Each expression is paired with its corresponding **LaTeX representation**, which serves as the target sequence for the recognition task.

**Handwritten strokes → Mathematical expression → LaTeX**

### Scope

This notebook is limited to **dataset exploration and analysis**. It does not perform model training, final image generation, or evaluation.

The purpose is to establish reliable dataset statistics and identify characteristics that will guide subsequent preprocessing and dataset-construction stages.

### Expected Outcome

By the end of the notebook, we will have:

- A verified inventory of CROHME expression files and splits.
- Dataset-level metadata and basic statistics.
- Geometric and LaTeX-related distributions.
- Examples of unusually long or complex expressions.
- Basic validation of the LaTeX annotation format.

These findings will provide the foundation for the subsequent preprocessing and model-development stages.

In [ ]:
# A small helper function is defined to 
# display project paths in a cleaner, project-relative format.

def display_path(path):
    return "/HMER" + str(path).split("/HMER", 1)[1]

In [ ]:
from pathlib import Path
import sys
import re
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.inkml_parser import parse_inkml


DATASET_ROOT = PROJECT_ROOT / "data" / "crohme2019"

TRAIN_DIR = DATASET_ROOT / "train"
VALID_DIR = DATASET_ROOT / "valid"
TEST_DIR = DATASET_ROOT / "test"

print(f"Project root: {display_path(PROJECT_ROOT)}")
print(f"Dataset root: {display_path(DATASET_ROOT)}")

#### Imports and Dataset Paths:

Import the required libraries and project utilities, then define the CROHME dataset directories for the training, validation, and test splits.


In [ ]:
train_files = sorted(TRAIN_DIR.glob("*.inkml"))
valid_files = sorted(VALID_DIR.glob("*.inkml"))
test_files = sorted(TEST_DIR.glob("*.inkml"))

print(f"Train: {len(train_files)}")
print(f"Valid: {len(valid_files)}")
print(f"Test:  {len(test_files)}")
print(f"Total: {len(train_files) + len(valid_files) + len(test_files)}")

#### Discover Dataset Files:

Collect all InkML expression files from the three dataset splits and report the number of samples available in each split and in total.


In [ ]:
def inspect_split(files, split_name):
    records = []

    for file_path in files:

        try:
            result = parse_inkml(file_path)

            strokes = result["strokes"]
            latex = result["latex"] or ""

            num_strokes = len(strokes)

            num_points = sum(
                len(stroke)
                for stroke in strokes
            )

            all_points = [
                point
                for stroke in strokes
                for point in stroke
            ]

            if all_points:
                xs = [p[0] for p in all_points]
                ys = [p[1] for p in all_points]

                width = max(xs) - min(xs)
                height = max(ys) - min(ys)

            else:
                width = 0
                height = 0

            records.append(
                {
                    "split": split_name,
                    "file": file_path.name,
                    "latex": latex,
                    "num_strokes": num_strokes,
                    "num_points": num_points,
                    "width": width,
                    "height": height,
                }
            )

        except Exception as exc:
            print(
                f"Failed to parse {file_path.name}: {exc}"
            )

    return pd.DataFrame(records)

### Inspect InkML Expressions

Parse each InkML file and extract the basic information required for dataset exploration:

- dataset split
- file name
- LaTeX expression
- number of strokes
- number of points
- expression width
- expression height

The extracted information is stored as records in a DataFrame.


In [ ]:
train_df = inspect_split(
    train_files,
    "train",
)

valid_df = inspect_split(
    valid_files,
    "valid",
)

test_df = inspect_split(
    test_files,
    "test",
)

#### Build Split-Level DataFrames:

Run the expression inspection separately for the training, validation, and test sets.


In [ ]:
df = pd.concat(
    [
        train_df,
        valid_df,
        test_df,
    ],
    ignore_index=True,
)

print(df.shape)

#### Combine the Dataset Metadata:

Combine the three split-level DataFrames into a single DataFrame for the remaining exploratory analysis.


In [ ]:
df[
    [
        "num_strokes",
        "num_points",
        "width",
        "height",
    ]
].describe()

#### Basic Geometric Statistics:

Inspect descriptive statistics for the number of strokes, number of points, expression width, and expression height.


In [ ]:
df["aspect_ratio"] = (
    df["width"] /
    df["height"].replace(0, np.nan)
)

#### Compute Expression Aspect Ratio:

Calculate the original aspect ratio of each handwritten expression from its extracted width and height.


In [ ]:
df["aspect_ratio"].describe()

#### Aspect Ratio Statistics:

Inspect the descriptive statistics of the expression aspect ratios.


In [ ]:
plt.figure(figsize=(10, 5))

plt.hist(
    df["aspect_ratio"].dropna(),
    bins=50,
)

plt.xlabel("Original expression aspect ratio")
plt.ylabel("Number of expressions")
plt.title("CROHME Expression Aspect Ratios")

plt.show()

#### Aspect Ratio Distribution:

Visualize how the original CROHME expression aspect ratios are distributed across the dataset.


In [ ]:
df["num_strokes"].describe()

#### Stroke Count Statistics:

Inspect the descriptive statistics of the number of strokes in each handwritten expression.


In [ ]:
plt.figure(figsize=(10, 5))

plt.hist(
    df["num_strokes"],
    bins=50,
)

plt.xlabel("Number of strokes")
plt.ylabel("Number of expressions")
plt.title("CROHME Stroke Count Distribution")

plt.show()

#### Stroke Count Distribution:

Visualize the distribution of stroke counts across the CROHME expressions.


In [ ]:
df["latex_length"] = df["latex"].str.len()

#### Compute LaTeX Expression Length:

Calculate the number of characters in each LaTeX expression.


In [ ]:
df["latex_length"].describe()

#### LaTeX Length Statistics:

Inspect the descriptive statistics of the LaTeX expression lengths.


In [ ]:
plt.figure(figsize=(10, 5))

plt.hist(
    df["latex_length"],
    bins=50,
)

plt.xlabel("LaTeX character length")
plt.ylabel("Number of expressions")
plt.title("CROHME LaTeX Length Distribution")

plt.show()

#### LaTeX Length Distribution:

Visualize the distribution of LaTeX character lengths across the dataset.


In [ ]:
df.nlargest(
    10,
    "latex_length"
)[
    [
        "file",
        "latex",
        "latex_length",
        "num_strokes",
    ]
]

#### Longest LaTeX Expressions:

Display the ten expressions with the greatest LaTeX character length together with their stroke counts.


In [ ]:
df["has_dollar"] = df["latex"].str.contains(
    r"\$",
    regex=True,
)

df["has_dollar"].value_counts()